# Tutorial 2: Train/Test Splits and Regularized Regression

**Big Data in Finance** | Dr Daniele Bianchi  
**Duration:** 1 hour

---

## Learning Objectives

By the end of this tutorial, you will be able to:

1. Split data correctly for time-series prediction
2. Fit OLS, Ridge, and Lasso regression using scikit-learn
3. Evaluate models with out-of-sample $R^2$
4. Use cross-validation to select the regularization parameter

---

## How This Tutorial Works

- **Run the code cells** in order (`Shift + Enter`)
- **🤖 AI Prompt exercises**: Practice using ChatGPT/Claude to write or explain code
- **🔧 Exercises**: Short tasks to test your understanding

> 💡 **Note (scope):** This tutorial uses a deliberately simplified setup for a one-hour session — a single chronological train/test split and a reduced set of predictors — so its numbers will not exactly reproduce the full walk-forward results in the lecture slides and notes.

---

## Part 1: Setup and Data Preparation

Run the cells below to load libraries and data.

In [ ]:
# =============================================================================
# Import libraries
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Scikit-learn: Machine learning library
# - LinearRegression: Ordinary Least Squares (OLS)
# - Ridge, Lasso: Regularized regression methods
# - TimeSeriesSplit: Cross-validation that respects time ordering
# - GridSearchCV: Automated hyperparameter tuning
# - StandardScaler: Standardizes features to mean=0, std=1
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# Set plot style for nicer visualizations
plt.style.use('seaborn-v0_8-whitegrid')

print("Libraries loaded!")

In [ ]:
# =============================================================================
# Load the Goyal-Welch-Zafirov dataset
# This dataset contains macroeconomic predictors for stock market returns
# =============================================================================

# Read the CSV file into a pandas DataFrame
df = pd.read_csv('Data_GWZ.csv')

# Convert the DATE column to datetime format and set as index
# This allows us to work with time-series data properly
df['DATE'] = pd.to_datetime(df['DATE'])
df = df.set_index('DATE')

print(f"Data shape: {df.shape}")
print(f"Period: {df.index[0].strftime('%Y-%m')} to {df.index[-1].strftime('%Y-%m')}")
df.head()

In [ ]:
# =============================================================================
# Define features (predictors) and target variable
# =============================================================================

# Target: MktRf = Market excess return (market return minus risk-free rate)
# This is what we want to predict
target = 'MktRf'

# Features: Macroeconomic variables that might predict returns
# d_p: dividend-price ratio (log)
# d_y: dividend yield (log)
# e_p: earnings-price ratio (log)
# b_m: book-to-market ratio
# tbl: Treasury bill rate (short-term interest rate)
# lty: Long-term government bond yield
# tms: Term spread (long rate - short rate)
# dfy: Default yield spread (corporate - government bonds)
# svar: Stock variance (realized volatility)
# infl: Inflation rate
feature_names = ['d_p', 'd_y', 'e_p', 'b_m', 'tbl', 'lty', 'tms', 'dfy', 'svar', 'infl']

print(f"Target: {target}")
print(f"Number of features: {len(feature_names)}")
print(f"Features: {feature_names}")

In [ ]:
# =============================================================================
# Create the prediction target: NEXT month's return
# =============================================================================

# IMPORTANT: We want to predict future returns, not current returns!
# shift(-1) moves the return column UP by one row
# So row t now contains the return from month t+1
#
# Example:
#   Before shift:  Jan features → Jan return
#   After shift:   Jan features → Feb return (what we want to predict!)

df['target'] = df[target].shift(-1)

# The last row now has NaN (no future return available), so drop it
df = df.dropna(subset=['target'])

# Separate features (X) and target (y)
X = df[feature_names]  # Predictor variables
y = df['target']       # What we're trying to predict

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

---

## Part 2: Train/Test Split

**Key rule:** For time series, always split chronologically — train on the past, test on the future.

In [ ]:
# =============================================================================
# Create chronological train/test split
# =============================================================================

# IMPORTANT: We do NOT use random splitting for time series!
# Random splits would create "look-ahead bias" - using future information
# to predict the past, which is impossible in real trading.
#
# Instead, we split by TIME:
# - Training set: First 80% of observations (older data)
# - Test set: Last 20% of observations (newer data)

# Calculate the split point (80% of data for training)
split_point = int(len(X) * 0.8)

# Split features
X_train = X.iloc[:split_point]   # First 80% of rows
X_test = X.iloc[split_point:]    # Last 20% of rows

# Split target (same split point)
y_train = y.iloc[:split_point]
y_test = y.iloc[split_point:]

# Print information about the split
print(f"Training: {X_train.index[0].strftime('%Y-%m')} to {X_train.index[-1].strftime('%Y-%m')} ({len(X_train)} obs)")
print(f"Test:     {X_test.index[0].strftime('%Y-%m')} to {X_test.index[-1].strftime('%Y-%m')} ({len(X_test)} obs)")

In [ ]:
# =============================================================================
# Visualize the train/test split
# =============================================================================

plt.figure(figsize=(12, 3))

# Plot training data in blue
plt.plot(y_train.index, y_train, label='Training', alpha=0.7)

# Plot test data in orange
plt.plot(y_test.index, y_test, label='Test', alpha=0.7)

# Add vertical line at the split point
plt.axvline(x=y_train.index[-1], color='red', linestyle='--', label='Split')

plt.xlabel('Date')
plt.ylabel('Return (%)')
plt.title('Chronological Train/Test Split')
plt.legend()
plt.tight_layout()
plt.show()

---

## Part 3: OLS Regression

### 3.1 Fit and Predict

In [ ]:
# =============================================================================
# Fit Ordinary Least Squares (OLS) regression
# =============================================================================

# OLS minimizes the sum of squared residuals:
# β_OLS = argmin Σ(y_i - x_i'β)²
#
# Scikit-learn workflow:
# 1. Create model object
# 2. Fit on training data
# 3. Predict on test data

# Step 1: Create the model
ols = LinearRegression()

# Step 2: Fit the model on training data
# This estimates the coefficients β
ols.fit(X_train, y_train)

# Step 3: Generate predictions on test data
# Predictions = X_test @ β_hat + intercept
y_pred_ols = ols.predict(X_test)

print("OLS model fitted!")
print(f"Number of coefficients: {len(ols.coef_)}")
print(f"Intercept: {ols.intercept_:.4f}")

### 3.2 Evaluate with Out-of-Sample $R^2$

In [ ]:
# =============================================================================
# Define the out-of-sample R² function
# =============================================================================

def oos_r2(y_true, y_pred, y_train_mean):
    """
    Calculate out-of-sample R² relative to the historical mean benchmark.
    
    Formula: R²_OOS = 1 - (SSE_model / SSE_mean)
    
    Where:
    - SSE_model = Σ(actual - predicted)²  [model's errors]
    - SSE_mean = Σ(actual - historical_mean)²  [benchmark errors]
    
    Interpretation:
    - R² > 0: Model beats the historical mean benchmark
    - R² = 0: Model equals the historical mean (no improvement)
    - R² < 0: Model is WORSE than just predicting the mean!
    
    Parameters:
    -----------
    y_true : array-like, actual observed values
    y_pred : array-like, model predictions
    y_train_mean : float, historical mean from training data
    
    Returns:
    --------
    float : out-of-sample R² value
    """
    # Sum of squared residuals (model errors)
    ss_res = np.sum((y_true - y_pred)**2)
    
    # Total sum of squares (benchmark errors using historical mean)
    ss_tot = np.sum((y_true - y_train_mean)**2)
    
    return 1 - ss_res / ss_tot

In [ ]:
# =============================================================================
# Calculate OOS R² for OLS
# =============================================================================

# The benchmark is the historical mean from training data
# This is what we would predict if we had no model
train_mean = y_train.mean()

# Calculate OOS R² for OLS
r2_ols = oos_r2(y_test.values, y_pred_ols, train_mean)

print(f"Historical mean (training): {train_mean:.4f}%")
print(f"OLS Out-of-Sample R²: {r2_ols:.4f} ({r2_ols*100:.2f}%)")

### 3.3 The Overfitting Problem

In [ ]:
# =============================================================================
# Compare in-sample vs out-of-sample performance
# =============================================================================

# In-sample: How well does the model fit the TRAINING data?
y_pred_train = ols.predict(X_train)
r2_in = 1 - np.sum((y_train - y_pred_train)**2) / np.sum((y_train - y_train.mean())**2)

# Out-of-sample: How well does it predict NEW data? (already calculated)

print(f"OLS In-Sample R²:      {r2_in:.4f} ({r2_in*100:.2f}%)")
print(f"OLS Out-of-Sample R²:  {r2_ols:.4f} ({r2_ols*100:.2f}%)")
print(f"\nGap: {(r2_in - r2_ols)*100:.2f} percentage points")
print("\n⚠️ Large gap between in-sample and out-of-sample = OVERFITTING")
print("   The model learned noise in the training data, not true patterns.")

---

## Part 4: Ridge and Lasso Regression

Regularization adds a penalty to prevent overfitting:
- **Ridge** ($L_2$): Shrinks all coefficients toward zero
- **Lasso** ($L_1$): Can set coefficients exactly to zero (variable selection)

### 4.1 Feature Scaling

⚠️ Regularized regression requires scaled features!

In [ ]:
# =============================================================================
# Standardize features for regularized regression
# =============================================================================

# WHY SCALE?
# Ridge/Lasso penalize the SIZE of coefficients (β)
# If features have different scales, the penalty is unfair:
# - Feature in % (0-100) → needs large β to have effect
# - Feature in decimals (0-1) → needs small β to have effect
# Scaling puts all features on equal footing.
#
# StandardScaler transforms each feature to: mean=0, std=1
# Formula: z = (x - mean) / std

scaler = StandardScaler()

# IMPORTANT: fit_transform on TRAINING data only!
# We learn the mean and std from training data
X_train_scaled = scaler.fit_transform(X_train)

# Then apply the SAME transformation to test data
# Using transform (not fit_transform) - uses training mean/std
# This prevents "data leakage" from test set
X_test_scaled = scaler.transform(X_test)

print("Features scaled!")
print(f"Training features - Mean: {X_train_scaled.mean(axis=0).round(2)}")
print(f"Training features - Std:  {X_train_scaled.std(axis=0).round(2)}")

### 4.2 Fit Ridge and Lasso

In [ ]:
# =============================================================================
# Fit Ridge Regression
# =============================================================================

# Ridge adds an L2 penalty to OLS:
# β_Ridge = argmin { Σ(y_i - x_i'β)² + α * Σβ_j² }
#                    ─────────────────   ────────
#                    OLS loss function   L2 penalty
#
# - α (alpha) controls the penalty strength
# - Larger α → more shrinkage → simpler model
# - α = 0 → back to OLS

ridge = Ridge(alpha=1.0)  # alpha is the regularization strength (λ in lecture)
ridge.fit(X_train_scaled, y_train)
y_pred_ridge = ridge.predict(X_test_scaled)

# Evaluate
r2_ridge = oos_r2(y_test.values, y_pred_ridge, train_mean)
print(f"Ridge (α=1.0) OOS R²: {r2_ridge:.4f} ({r2_ridge*100:.2f}%)")

In [ ]:
# =============================================================================
# Fit Lasso Regression
# =============================================================================

# Lasso adds an L1 penalty to OLS:
# β_Lasso = argmin { Σ(y_i - x_i'β)² + α * Σ|β_j| }
#                    ─────────────────   ─────────
#                    OLS loss function   L1 penalty
#
# KEY DIFFERENCE from Ridge:
# - L1 penalty can shrink coefficients EXACTLY to zero
# - This performs automatic variable selection
# - Some features are "dropped" from the model

lasso = Lasso(alpha=0.01)  # Lasso often needs smaller alpha than Ridge
lasso.fit(X_train_scaled, y_train)
y_pred_lasso = lasso.predict(X_test_scaled)

# Evaluate
r2_lasso = oos_r2(y_test.values, y_pred_lasso, train_mean)
print(f"Lasso (α=0.01) OOS R²: {r2_lasso:.4f} ({r2_lasso*100:.2f}%)")

In [ ]:
# =============================================================================
# Compare all models so far
# =============================================================================

print("Model Comparison (OOS R²):")
print("="*40)
print(f"OLS:           {r2_ols*100:+.2f}%")
print(f"Ridge (α=1.0): {r2_ridge*100:+.2f}%")
print(f"Lasso (α=0.01):{r2_lasso*100:+.2f}%")

### 4.3 Lasso Variable Selection

In [ ]:
# =============================================================================
# Examine which features Lasso selected (non-zero coefficients)
# =============================================================================

# Create a DataFrame showing each feature's coefficient
lasso_coefs = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': lasso.coef_,
    'Selected': lasso.coef_ != 0  # True if coefficient is non-zero
})

print("Lasso Coefficients:")
print(lasso_coefs.to_string(index=False))
print(f"\nFeatures selected: {sum(lasso.coef_ != 0)} out of {len(feature_names)}")
print("\n💡 Lasso automatically dropped features it considers unimportant!")

### 🤖 AI Prompt Exercise 1: Compare Coefficients

**Task:** Ask an AI to write code that creates a bar chart comparing OLS, Ridge, and Lasso coefficients.

**Copy this prompt:**

```
Write Python code to create a grouped bar chart comparing regression coefficients.

I have:
- feature_names: list of 10 feature names
- Three fitted models on scaled data: ols_scaled, ridge, lasso
- Each model has a .coef_ attribute (numpy array of coefficients)

First fit OLS on scaled data:
  ols_scaled = LinearRegression()
  ols_scaled.fit(X_train_scaled, y_train)

Then create a bar chart with:
- Features on x-axis
- Three bars per feature (OLS, Ridge, Lasso)
- Legend and title
- Figure size (12, 5)
```

**Paste the AI-generated code below and run it:**

In [ ]:
# Paste AI-generated code here:


---

## Part 5: Choosing $\lambda$ with Cross-Validation

How do we pick the best regularization strength? Use **time-series cross-validation**.

In [ ]:
# =============================================================================
# Set up Time-Series Cross-Validation
# =============================================================================

# Standard K-fold CV randomly shuffles data - BAD for time series!
# It would use future data to predict the past (look-ahead bias)
#
# TimeSeriesSplit respects time ordering:
# Fold 1: Train on [1...100], Validate on [101...120]
# Fold 2: Train on [1...120], Validate on [121...140]
# Fold 3: Train on [1...140], Validate on [141...160]
# etc.
#
# Training window EXPANDS, validation always comes AFTER training

tscv = TimeSeriesSplit(n_splits=5)  # 5 folds

# Visualize the folds
print("Time-Series CV Folds:")
print("="*50)
for i, (train_idx, val_idx) in enumerate(tscv.split(X_train_scaled), 1):
    print(f"Fold {i}: Train = {len(train_idx)} obs, Validate = {len(val_idx)} obs")

### 🤖 AI Prompt Exercise 2: Write a Cross-Validation Loop

**Task:** Ask an AI to write code that finds the best alpha for Ridge using CV.

**Copy this prompt:**

```
Write Python code to find the best alpha for Ridge regression using time-series cross-validation.

I have:
- X_train_scaled: scaled training features (numpy array)
- y_train: training target (pandas Series)
- tscv: TimeSeriesSplit object with 5 folds

Test these alpha values: [0.01, 0.1, 1, 10, 100]

For each alpha:
1. Loop through the 5 CV folds
2. Fit Ridge on training fold, predict on validation fold
3. Calculate mean squared error (MSE)
4. Average MSE across folds

Print results for each alpha and identify the best one (lowest MSE).
```

**Paste and run:**

In [ ]:
# Paste AI-generated code here:


### 5.2 Using GridSearchCV (Built-in Method)

scikit-learn has a built-in function that does this automatically:

In [ ]:
# =============================================================================
# Use GridSearchCV for automated hyperparameter tuning
# =============================================================================

# GridSearchCV automates the cross-validation loop:
# 1. Define a grid of hyperparameters to try
# 2. For each combination, perform CV
# 3. Select the best hyperparameters

# Define the parameter grid
# Dictionary: {'parameter_name': [list of values to try]}
param_grid = {'alpha': [0.01, 0.1, 1, 10, 100]}

# Create GridSearchCV object
grid_search = GridSearchCV(
    Ridge(),                            # The model to tune
    param_grid,                         # Parameters to search
    cv=tscv,                            # Cross-validation strategy (time-series!)
    scoring='neg_mean_squared_error'    # Metric to optimize (negative because sklearn maximizes)
)

# Fit performs the entire CV search
grid_search.fit(X_train_scaled, y_train)

# Results
print(f"Best alpha: {grid_search.best_params_['alpha']}")
print(f"Best CV score (neg MSE): {grid_search.best_score_:.6f}")

### 5.3 Final Evaluation

In [ ]:
# =============================================================================
# Evaluate the best model on the test set
# =============================================================================

# Get the best model from GridSearchCV
# This is already fitted on all training data with best hyperparameters
best_ridge = grid_search.best_estimator_

# Make predictions on test set
y_pred_best = best_ridge.predict(X_test_scaled)

# Calculate OOS R²
r2_best = oos_r2(y_test.values, y_pred_best, train_mean)

print(f"Best Ridge (α={grid_search.best_params_['alpha']}) OOS R²: {r2_best:.4f} ({r2_best*100:.2f}%)")

---

## Part 6: Summary

### Results Comparison

In [ ]:
# =============================================================================
# Create final summary table
# =============================================================================

results = pd.DataFrame({
    'Model': [
        'Historical Mean',           # Benchmark
        'OLS',                        # No regularization
        'Ridge (α=1)',                # Fixed alpha
        f'Ridge (best α={grid_search.best_params_["alpha"]})',  # CV-tuned
        'Lasso (α=0.01)'              # L1 regularization
    ],
    'OOS R² (%)': [
        0.00,               # Historical mean is the benchmark (R²=0 by definition)
        r2_ols*100,
        r2_ridge*100,
        r2_best*100,
        r2_lasso*100
    ]
})

print("Model Comparison:")
print("="*45)
print(results.to_string(index=False))

### 🤖 AI Prompt Exercise 3: Interpret Your Results

**Task:** Ask an AI to help interpret your findings.

**Copy this prompt (fill in your actual numbers):**

```
I ran OLS, Ridge, and Lasso regression to predict stock market returns. Here are my out-of-sample R² results:

- OLS: [YOUR VALUE]%
- Ridge: [YOUR VALUE]%  
- Lasso: [YOUR VALUE]%

Questions:
1. Why might OLS perform poorly compared to regularized methods?
2. Is a 1-2% R² actually meaningful for return prediction?
3. What does it mean if R² is negative?

Keep the answer concise (5-6 sentences).
```

**Write the key insights you learned:**

*Your notes:*



---

## 🔧 Exercise: Try Lasso with CV

Repeat the GridSearchCV process for Lasso instead of Ridge. Which alpha works best?

In [ ]:
# Your code here (hint: change Ridge() to Lasso() and adjust alpha values)
# Lasso often needs smaller alpha values, try: [0.001, 0.005, 0.01, 0.05, 0.1]


---

## Key Takeaways

1. **Always split chronologically** for time series data

2. **OLS overfits** when there are many predictors — good in-sample, poor out-of-sample

3. **Ridge** shrinks all coefficients toward zero, reducing variance

4. **Lasso** can set coefficients exactly to zero (variable selection)

5. **Use time-series CV** to select the regularization parameter $\lambda$

6. **Even 1-2% OOS $R^2$** can be economically significant for returns

---

## Next Week

**Topic:** Tree-based methods (Random Forests, Gradient Boosting)

**Reading:** James et al. (2023), Chapter 8